In [11]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from optional_fine_tune import BertClassification, DfToDataset

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Используемое устройство: cuda


In [13]:
test = pd.read_csv("data/test.csv")
train = pd.read_csv("data/train.csv")

test_texts = test['comment_text']
target_columns = train.columns[2:].tolist()

num_classes = 6
test_targets = np.zeros((len(test_texts), 6))

In [14]:
test_dataset = DfToDataset(test_texts, test_targets, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [15]:
print("Загрузка весов моделей с диска...")
models = []

for fold in range(5):
    model = BertClassification(MODEL_NAME)
    model_path = f"models/best_bert_fold_{fold + 1}.pt"

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Файл весов {model_path} не найден! Убедись, что он лежит в папке со скриптом.")

    weights = torch.load(model_path, map_location=device)
    model.load_state_dict(weights)
    model.to(device)
    model.eval()  # Отключаем Dropout!
    models.append(model)
print("Все 5 моделей успешно загружены в память.")

Загрузка весов моделей с диска...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7517.62it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\delux\AppData\Local\Temp\ipykernel_10888\3121940317.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, t

Все 5 моделей успешно загружены в память.


In [16]:
print("Запуск предсказания на тестовых данных...")
all_blend_preds = []

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        batch_probs = []

        for model in models:
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(logits)
            batch_probs.append(probs.cpu().numpy())

        avg_probs = np.mean(batch_probs, axis=0)
        all_blend_preds.extend(avg_probs)

        if i % 50 == 0:
            print(f"Обработано батчей: {i}/{len(test_loader)} ")

final_matrix = np.vstack(all_blend_preds)
submission = pd.DataFrame(final_matrix, columns=target_columns)
submission.insert(0, 'id', test['id'].values)

submission.to_csv("results/submission_ensemble.csv", index=False)

Запуск предсказания на тестовых данных...
Обработано батчей: 0/2394 
Обработано батчей: 50/2394 
Обработано батчей: 100/2394 
Обработано батчей: 150/2394 
Обработано батчей: 200/2394 
Обработано батчей: 250/2394 
Обработано батчей: 300/2394 
Обработано батчей: 350/2394 
Обработано батчей: 400/2394 
Обработано батчей: 450/2394 
Обработано батчей: 500/2394 
Обработано батчей: 550/2394 
Обработано батчей: 600/2394 
Обработано батчей: 650/2394 
Обработано батчей: 700/2394 
Обработано батчей: 750/2394 
Обработано батчей: 800/2394 
Обработано батчей: 850/2394 
Обработано батчей: 900/2394 
Обработано батчей: 950/2394 
Обработано батчей: 1000/2394 
Обработано батчей: 1050/2394 
Обработано батчей: 1100/2394 
Обработано батчей: 1150/2394 
Обработано батчей: 1200/2394 
Обработано батчей: 1250/2394 
Обработано батчей: 1300/2394 
Обработано батчей: 1350/2394 
Обработано батчей: 1400/2394 
Обработано батчей: 1450/2394 
Обработано батчей: 1500/2394 
Обработано батчей: 1550/2394 
Обработано батчей: 16